<a href="https://colab.research.google.com/github/Khamidrees/Khalid-Muhammad-Idris-DSN-HACKATHON-/blob/main/Submission_Exact_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)

train = pd.read_csv('/mnt/data/train.csv')
test = pd.read_csv('/mnt/data/test.csv')
sample_submission = pd.read_csv('/mnt/data/sample_submission.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


Train shape: (6818, 13)
Test shape: (1705, 12)


,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format,total_sales
0,row_00000,PRD-PRFP9S,14.252,Low Fat,0.0271,Frozen Foods,81.37,STORE-AGY,45,Large,Tier_3,Standard Supermarket,1764.98
1,row_00001,PRD-PXXK71,7.698,Low Fat,0.0720,HEALTH AND HYGIENE,42.05,STORE-YLW,35,Small,Tier_1,Standard Supermarket,342.13
2,row_00002,PRD-V5MOIJ,14.264,Regular,0.0421,Canned,41.35,STORE-89Z,33,Medium,Tier_1,Standard Supermarket,378.85
3,row_00003,PRD-UN5Z3J,NaN,Regular,0.0449,soft drinks,174.35,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket,5595.72
4,row_00004,PRD-6RDQYB,10.338,Regular,0.0120,meat,203.06,STORE-9RG,28,Small,Tier_2,Standard Supermarket,2375.36


In [ ]:
train.dtypes


id                      object
product_code            object
product_weight_kg      float64
fat_content             object
shelf_visibility       float64
product_category        object
product_price          float64
store_code              object
store_age_years          int64
store_size              object
store_location_tier     object
store_format            object
total_sales            float64
dtype: object

In [ ]:
print("Missing values (train):")
print(train.isnull().sum())
print()
print("Missing values (test):")
print(test.isnull().sum())


Missing values (train):
id                        0
product_code              0
product_weight_kg      1225
fat_content               0
shelf_visibility          0
product_category          0
product_price             0
store_code                0
store_age_years           0
store_size             1919
store_location_tier       0
store_format              0
total_sales               0
dtype: int64

Missing values (test):
id                       0
product_code             0
product_weight_kg      306
fat_content              0
shelf_visibility         0
product_category         0
product_price            0
store_code               0
store_age_years          0
store_size             491
store_location_tier      0
store_format             0
dtype: int64


In [ ]:
def clean_category(df):
    df = df.copy()
    df['fat_content'] = df['fat_content'].str.strip()
    df['product_category'] = df['product_category'].str.strip().str.lower()

    cat_map = {
        'frozen foods':'Frozen Foods','health and hygiene':'Health and Hygiene','canned':'Canned',
        'soft drinks':'Soft Drinks','meat':'Meat','snack foods':'Snack Foods','baking goods':'Baking Goods',
        'household':'Household','dairy':'Dairy','fruits and vegetables':'Fruits and Vegetables',
        'breakfast':'Breakfast','breads':'Breads','others':'Others','seafood':'Seafood',
        'hard drinks':'Hard Drinks','starchy foods':'Starchy Foods'
    }
    df['product_category'] = df['product_category'].map(cat_map)
    return df

train = clean_category(train)
test = clean_category(test)

print("Cleaned categories:", sorted(train.product_category.unique()))
print("Fat content values:", train.fat_content.unique())


Cleaned categories: ['Baking Goods', 'Breads', 'Breakfast', 'Canned', 'Dairy', 'Frozen Foods', 'Fruits and Vegetables', 'Hard Drinks', 'Health and Hygiene', 'Household', 'Meat', 'Others', 'Seafood', 'Snack Foods', 'Soft Drinks', 'Starchy Foods']
Fat content values: ['Low Fat' 'Regular']


In [ ]:
# Combine train and test for consistent imputation (no leakage of the target)
all_df = pd.concat(
    [train.assign(_src='train'), test.assign(_src='test')],
    ignore_index=True, sort=False
)

# --- Impute product_weight_kg ---
weight_by_code = all_df.groupby('product_code')['product_weight_kg'].transform('mean')
all_df['product_weight_kg'] = all_df['product_weight_kg'].fillna(weight_by_code)

weight_by_cat = all_df.groupby('product_category')['product_weight_kg'].transform('mean')
all_df['product_weight_kg'] = all_df['product_weight_kg'].fillna(weight_by_cat)

# --- Impute store_size ---
store_size_map = (
    all_df.dropna(subset=['store_size'])
    .drop_duplicates('store_code')
    .set_index('store_code')['store_size']
    .to_dict()
)

missing_stores = set(all_df.store_code.unique()) - set(store_size_map.keys())
print("Stores with no size info at all:", missing_stores)

for s in missing_stores:
    fmt = all_df.loc[all_df.store_code == s, 'store_format'].iloc[0]
    similar = all_df[all_df.store_format == fmt]['store_size'].mode()
    store_size_map[s] = similar.iloc[0] if len(similar) else 'Medium'

print("Final store size map:", store_size_map)
all_df['store_size'] = all_df['store_code'].map(store_size_map)

print()
print("Remaining missing values:")
print(all_df.drop(columns=['total_sales']).isnull().sum().sum(), "cells (excluding target)")


Stores with no size info at all: {'STORE-JOR', 'STORE-OYG', 'STORE-DKU'}
Final store size map: {'STORE-AGY': 'Large', 'STORE-YLW': 'Small', 'STORE-89Z': 'Medium', 'STORE-7WS': 'Medium', 'STORE-9RG': 'Small', 'STORE-HL7': 'Medium', 'STORE-T5G': 'Small', 'STORE-JOR': 'Small', 'STORE-OYG': 'Small', 'STORE-DKU': 'Small'}

Remaining missing values:
0 cells (excluding target)


In [ ]:
all_df['price_per_kg'] = all_df['product_price'] / all_df['product_weight_kg']

store_avg_price = all_df.groupby('store_code')['product_price'].transform('mean')
all_df['store_avg_price'] = store_avg_price

cat_avg_price = all_df.groupby('product_category')['product_price'].transform('mean')
all_df['cat_avg_price'] = cat_avg_price
all_df['price_vs_cat_avg'] = all_df['product_price'] - all_df['cat_avg_price']

cat_avg_vis = all_df.groupby('product_category')['shelf_visibility'].transform('mean')
all_df['vis_vs_cat_avg'] = all_df['shelf_visibility'] - cat_avg_vis

all_df['store_age_bin'] = pd.cut(
    all_df['store_age_years'], bins=[0, 25, 30, 40, 50],
    labels=['<=25', '26-30', '31-40', '41+']
)

train_fe = all_df[all_df._src == 'train'].drop(columns=['_src']).reset_index(drop=True)
test_fe = all_df[all_df._src == 'test'].drop(columns=['_src', 'total_sales']).reset_index(drop=True)

print(train_fe.shape, test_fe.shape)
train_fe.head()


(6818, 19) (1705, 18)


,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format,total_sales,price_per_kg,store_avg_price,cat_avg_price,price_vs_cat_avg,vis_vs_cat_avg,store_age_bin
0,row_00000,PRD-PRFP9S,14.252000,Low Fat,0.0271,Frozen Foods,81.37,STORE-AGY,45,Large,Tier_3,Standard Supermarket,1764.98,5.709374,141.355376,138.600864,-57.230864,-0.038665,41+
1,row_00001,PRD-PXXK71,7.698000,Low Fat,0.0720,Health and Hygiene,42.05,STORE-YLW,35,Small,Tier_1,Standard Supermarket,342.13,5.462458,142.169022,130.811635,-88.761635,0.016847,31-40
2,row_00002,PRD-V5MOIJ,14.264000,Regular,0.0421,Canned,41.35,STORE-89Z,33,Medium,Tier_1,Standard Supermarket,378.85,2.898906,140.385796,139.720924,-98.370924,-0.025905,31-40
3,row_00003,PRD-UN5Z3J,12.899667,Regular,0.0449,Soft Drinks,174.35,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket,5595.72,13.515853,139.763176,131.574180,42.775820,-0.019044,41+
4,row_00004,PRD-6RDQYB,10.338000,Regular,0.0120,Meat,203.06,STORE-9RG,28,Small,Tier_2,Standard Supermarket,2375.36,19.642097,143.052656,139.876212,63.183788,-0.050442,26-30


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42

target = 'total_sales'
drop_cols = ['id', 'product_code', 'store_code', target]
feature_cols = [c for c in train_fe.columns if c not in drop_cols]

cat_cols = train_fe[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]
print("Categorical:", cat_cols)
print("Numeric:", num_cols)

X = train_fe[feature_cols].copy()
y = train_fe[target].copy()
X_test = test_fe[feature_cols].copy()

for c in cat_cols:
    X[c] = X[c].astype(str)
    X_test[c] = X_test[c].astype(str)

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, np.clip(y_pred, 0, None)))

def rmse_cv(model, X, y):
    scores = []
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        scores.append(rmse(y_val, pred))
    return np.mean(scores), np.std(scores)


Categorical: ['fat_content', 'product_category', 'store_size', 'store_location_tier', 'store_format', 'store_age_bin']
Numeric: ['product_weight_kg', 'shelf_visibility', 'product_price', 'store_age_years', 'price_per_kg', 'store_avg_price', 'cat_avg_price', 'price_vs_cat_avg', 'vis_vs_cat_avg']


In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge

models = {
    'Ridge': Ridge(alpha=1.0),
    'GBR': GradientBoostingRegressor(random_state=RANDOM_STATE, n_estimators=300, learning_rate=0.05, max_depth=3),
    'HistGB': HistGradientBoostingRegressor(random_state=RANDOM_STATE, max_iter=300, learning_rate=0.05, max_depth=4),
    'RandomForest': RandomForestRegressor(n_estimators=400, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1),
}

for name, m in models.items():
    mean_s, std_s = rmse_cv(m, X, y)
    print(f"{name}: RMSE = {mean_s:.2f} (+/- {std_s:.2f})")


Ridge: RMSE = 1233.13 (+/- 23.12)
GBR: RMSE = 1099.88 (+/- 24.22)
HistGB: RMSE = 1097.84 (+/- 23.49)
RandomForest: RMSE = 1094.85 (+/- 24.09)


In [ ]:
rf_configs = [
    dict(n_estimators=300, max_depth=6, min_samples_leaf=8, max_features=0.7),
    dict(n_estimators=400, max_depth=7, min_samples_leaf=4, max_features='sqrt'),
    dict(n_estimators=300, max_depth=8, min_samples_leaf=15, max_features=0.5),
    dict(n_estimators=500, max_depth=6, min_samples_leaf=10, max_features=0.7),
]

best_rf_score, best_rf_cfg = np.inf, None
for cfg in rf_configs:
    m = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **cfg)
    s, _ = rmse_cv(m, X, y)
    print("RF", cfg, "->", round(s, 2))
    if s < best_rf_score:
        best_rf_score, best_rf_cfg = s, cfg

print("\nBest RF config:", best_rf_cfg, "RMSE:", round(best_rf_score, 2))


RF {'n_estimators': 300, 'max_depth': 6, 'min_samples_leaf': 8, 'max_features': 0.7} -> 1085.93
RF {'n_estimators': 400, 'max_depth': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt'} -> 1086.16
RF {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 15, 'max_features': 0.5} -> 1084.45
RF {'n_estimators': 500, 'max_depth': 6, 'min_samples_leaf': 10, 'max_features': 0.7} -> 1085.43

Best RF config: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 15, 'max_features': 0.5} RMSE: 1084.45


In [ ]:
hgb_configs = [
    dict(max_iter=300, learning_rate=0.05, max_depth=4, max_leaf_nodes=31, min_samples_leaf=20),
    dict(max_iter=500, learning_rate=0.03, max_depth=4, max_leaf_nodes=31, min_samples_leaf=20),
    dict(max_iter=300, learning_rate=0.05, max_depth=3, max_leaf_nodes=15, min_samples_leaf=15),
    dict(max_iter=200, learning_rate=0.08, max_depth=5, max_leaf_nodes=63, min_samples_leaf=30),
]

best_hgb_score, best_hgb_cfg = np.inf, None
for cfg in hgb_configs:
    m = HistGradientBoostingRegressor(random_state=RANDOM_STATE, **cfg)
    s, _ = rmse_cv(m, X, y)
    print("HGB", cfg, "->", round(s, 2))
    if s < best_hgb_score:
        best_hgb_score, best_hgb_cfg = s, cfg

print("\nBest HGB config:", best_hgb_cfg, "RMSE:", round(best_hgb_score, 2))


HGB {'max_iter': 300, 'learning_rate': 0.05, 'max_depth': 4, 'max_leaf_nodes': 31, 'min_samples_leaf': 20} -> 1097.84
HGB {'max_iter': 500, 'learning_rate': 0.03, 'max_depth': 4, 'max_leaf_nodes': 31, 'min_samples_leaf': 20} -> 1098.24
HGB {'max_iter': 300, 'learning_rate': 0.05, 'max_depth': 3, 'max_leaf_nodes': 15, 'min_samples_leaf': 15} -> 1090.95
HGB {'max_iter': 200, 'learning_rate': 0.08, 'max_depth': 5, 'max_leaf_nodes': 63, 'min_samples_leaf': 30} -> 1105.6

Best HGB config: {'max_iter': 300, 'learning_rate': 0.05, 'max_depth': 3, 'max_leaf_nodes': 15, 'min_samples_leaf': 15} RMSE: 1090.95


In [ ]:
def rmse_cv_blend(w_rf, rf_cfg, hgb_cfg, X, y):
    scores = []
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **rf_cfg).fit(X_tr, y_tr)
        hgb = HistGradientBoostingRegressor(random_state=RANDOM_STATE, **hgb_cfg).fit(X_tr, y_tr)
        pred = w_rf * rf.predict(X_val) + (1 - w_rf) * hgb.predict(X_val)
        scores.append(rmse(y_val, pred))
    return np.mean(scores)

for w in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
    s = rmse_cv_blend(w, best_rf_cfg, best_hgb_cfg, X, y)
    print(f"weight_rf={w}: RMSE={s:.2f}")


weight_rf=0.3: RMSE=1086.26
weight_rf=0.4: RMSE=1085.21
weight_rf=0.5: RMSE=1084.43
weight_rf=0.6: RMSE=1083.90
weight_rf=0.7: RMSE=1083.64
weight_rf=0.8: RMSE=1083.65
weight_rf=1.0: RMSE=1084.45


In [ ]:
rf_full = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_rf_cfg).fit(X, y)
importances = pd.Series(rf_full.feature_importances_, index=X.columns).sort_values(ascending=False)
importances


product_price          0.302520
store_format           0.282798
price_vs_cat_avg       0.207232
store_age_years        0.050434
store_avg_price        0.049965
price_per_kg           0.044373
store_size             0.014609
product_weight_kg      0.008672
shelf_visibility       0.008587
vis_vs_cat_avg         0.007919
store_location_tier    0.007523
store_age_bin          0.005277
cat_avg_price          0.005221
product_category       0.003933
fat_content            0.000937
dtype: float64

In [ ]:
W_RF = 0.7

final_rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_rf_cfg).fit(X, y)
final_hgb = HistGradientBoostingRegressor(random_state=RANDOM_STATE, **best_hgb_cfg).fit(X, y)

pred = W_RF * final_rf.predict(X_test) + (1 - W_RF) * final_hgb.predict(X_test)
pred = np.clip(pred, 0, None)  # sales can't be negative

submission = pd.DataFrame({'id': test_fe['id'], 'total_sales': np.round(pred, 2)})
submission.head(10)


,id,total_sales
0,row_00009,2912.43
1,row_00015,4443.41
2,row_00019,3118.22
3,row_00020,3162.59
4,row_00023,2555.71
5,row_00026,373.05
6,row_00027,3935.37
7,row_00033,3029.71
8,row_00034,348.53
9,row_00037,1993.83


In [ ]:
assert list(submission.columns) == list(sample_submission.columns)
assert len(submission) == len(sample_submission)
assert set(submission.id) == set(sample_submission.id)

submission.to_csv('submission.csv', index=False)
print("Saved submission.csv:", submission.shape)
submission.describe()


Saved submission.csv: (1705, 2)


,total_sales
count,1705.000000
mean,2190.018551
std,1305.427380
min,56.610000
25%,1185.960000
50%,2077.940000
75%,3061.440000
max,6235.850000
